# From Pixels to Patches: Vision Transformers for Medical Image Understanding

**AI for Medicine Bootcamp — Workshop (3 Hours)**

In this workshop we apply the same intellectual spine from earlier sessions — **tokenization, embeddings, attention, pretraining, transfer learning** — to medical images. Instead of words or DNA bases, we work with image patches. Instead of masked language modeling, we use supervised classification. And we extend from single-modality (image-only) to multi-modal (image + text) understanding.

**What you will build:**
1. Explore a real radiology image-caption dataset (ROCOv2)
2. Tokenize images into patches — the visual equivalent of words
3. Build a Vision Transformer (ViT) from scratch and train it
4. Fine-tune a pretrained ViT with LoRA for medical image classification
5. Connect vision and language with contrastive learning (CLIP / BiomedCLIP)
6. Interpret what the model "sees" through attention visualization

---

**Table of Contents**

| Section | Topic | Time |
|---------|-------|------|
| 0 | Setup and Environment | — |
| 1 | The Big Picture — Why Transformers for Vision? | 0:00–0:15 |
| 2 | Data Tour — Radiology Images Meet Text | 0:15–0:35 |
| 3 | Image Tokenization — Seeing in Patches | 0:35–1:05 |
| 4 | Building a ViT from Scratch | 1:05–1:50 |
| 5 | Pre-trained ViT and Transfer Learning | 1:50–2:15 |
| 6 | Multimodal — From ViT to Vision-Language Models | 2:15–2:45 |
| 7 | Interpretation — What Does ViT See? | 2:45–2:55 |
| 8 | Wrap-Up | 2:55–3:00 |

> **Colab GPU:** Go to *Runtime → Change runtime type → T4 GPU* before running.

In [ ]:
# ── Section 0: Setup and Environment ──────────────────────────────────────────
import sys, subprocess

def pip_install(packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

pip_install([
    "-U", "transformers", "datasets", "peft", "accelerate",
    "timm", "open_clip_torch", "scikit-learn", "Pillow",
    "torchvision", "huggingface_hub",
])

import torch, torch.nn as nn, torch.nn.functional as F
import transformers, datasets, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import torchvision.transforms as T
from PIL import Image
from IPython.display import display, HTML
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix, classification_report,
)
from sklearn.manifold import TSNE
from collections import Counter
import warnings, re, math
warnings.filterwarnings("ignore")

print(f"torch: {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"numpy: {np.__version__}")

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
    print("GPU: Apple Silicon (MPS)")
else:
    device = torch.device("cpu")
    print("No GPU — running on CPU (training will be slower)")

USE_PRECOMPUTED = False

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

plt.rcParams.update({
    "figure.figsize": (10, 5),
    "font.size": 13,
    "axes.titlesize": 15,
    "axes.labelsize": 13,
    "text.color": "black",
    "axes.labelcolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
})
sns.set_style("whitegrid")

In [ ]:
def plot_training_curves(train_losses, val_metrics, metric_name="Accuracy"):
    """Plot training loss and validation metric side-by-side."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(train_losses, linewidth=2, color="steelblue")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.set_title("Training Loss")
    ax2.plot(val_metrics, linewidth=2, color="coral")
    ax2.set_xlabel("Epoch"); ax2.set_ylabel(metric_name); ax2.set_title(f"Validation {metric_name}")
    plt.tight_layout(); plt.show()


def show_image_grid(images, titles=None, ncols=4, figsize=(16, 12)):
    """Display a grid of PIL images with optional titles."""
    n = len(images)
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes = np.array(axes).flatten()
    for i, ax in enumerate(axes):
        if i < n:
            ax.imshow(images[i])
            if titles and i < len(titles):
                ax.set_title(titles[i], fontsize=9, wrap=True)
        ax.axis("off")
    plt.tight_layout(); plt.show()


def display_comparison_table(rows, columns, title=""):
    """Render an HTML table for side-by-side comparison."""
    html = f'<h3 style="color:black;">{title}</h3>' if title else ""
    html += '<table style="width:100%; border-collapse:collapse; font-size:13px; color:black;">'
    html += "<tr>" + "".join(
        f'<th style="border:1px solid #ccc; padding:8px; background:#f0f0f0; color:black;">{c}</th>'
        for c in columns
    ) + "</tr>"
    for row in rows:
        html += "<tr>" + "".join(
            f'<td style="border:1px solid #ccc; padding:8px; vertical-align:top; white-space:pre-wrap; color:black;">{cell}</td>'
            for cell in row
        ) + "</tr>"
    html += "</table>"
    display(HTML(html))


IMG_SIZE = 224
PATCH_SIZE = 16
NUM_PATCHES = (IMG_SIZE // PATCH_SIZE) ** 2  # 196

train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
val_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
unnormalize = T.Compose([
    T.Normalize(mean=[0, 0, 0], std=[1/0.229, 1/0.224, 1/0.225]),
    T.Normalize(mean=[-0.485, -0.456, -0.406], std=[1, 1, 1]),
])

print("Helpers ready.")

---

## Section 1: The Big Picture — Why Transformers for Vision?

### NLP ↔ Vision: Concept Mapping

| NLP (Previous Sessions) | Vision (Today) |
|---|---|
| Words / subwords | **Image patches** (16×16 pixel regions) |
| BPE tokenizer | **Patch embedding** (Conv2d projection) |
| `[CLS]` token | `[CLS]` token (same idea!) |
| Positional encoding | **Learned 2D position embeddings** |
| Transformer encoder | Transformer encoder (identical!) |
| Text classification | **Image classification** |
| BERT | **ViT** (Vision Transformer) |
| CLIP text encoder | **CLIP image encoder** (ViT) |

The key insight of the Vision Transformer (ViT) is:

> **"An image is worth 16×16 words."** — Dosovitskiy et al., 2020

Instead of designing specialized convolutional architectures for images, ViT simply:
1. Cuts the image into a grid of patches
2. Flattens each patch into a vector (like a "word")
3. Feeds the sequence of patch vectors into a standard Transformer

```
┌─────────────────────────────────────────────────────────────────┐
│                        ViT Architecture                         │
│                                                                 │
│  Image ──→ [Split into    ──→ [Linear      ──→ [+ Position     │
│             16×16 patches]     Projection]      Embeddings]     │
│                                                                 │
│         ──→ [CLS] + Patch Tokens                                │
│         ──→ [Transformer Encoder × L layers]                    │
│         ──→ [CLS output] ──→ [MLP Head] ──→ Class prediction   │
└─────────────────────────────────────────────────────────────────┘
```

### CNN vs ViT

| | CNN | ViT |
|---|---|---|
| **Unit of processing** | Local pixel neighborhoods | Global patch sequence |
| **Inductive bias** | Translation equivariance built-in | Learned from data |
| **Receptive field** | Grows slowly through layers | Global from layer 1 |
| **Scaling** | Diminishing returns at scale | Scales well with more data |
| **Interpretability** | GradCAM on feature maps | Attention maps on patches |

In [ ]:
# Viz 1 — CNN vs ViT: How they "see" an image
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# CNN: local sliding filters
ax = axes[0]
ax.set_xlim(0, 7); ax.set_ylim(0, 7); ax.set_aspect("equal")
for i in range(7):
    for j in range(7):
        color = "#dce6f1" if not (2 <= i <= 4 and 2 <= j <= 4) else "#4a86c8"
        alpha = 0.3 if not (2 <= i <= 4 and 2 <= j <= 4) else 0.8
        rect = plt.Rectangle((i, j), 1, 1, facecolor=color, edgecolor="gray",
                              linewidth=0.5, alpha=alpha)
        ax.add_patch(rect)
highlight = plt.Rectangle((2, 2), 3, 3, facecolor="none", edgecolor="red", linewidth=3)
ax.add_patch(highlight)
ax.set_title("CNN: Local 3×3 Filter\n(slides across image)", fontsize=14, fontweight="bold")
ax.annotate("receptive field\ngrows slowly", xy=(3.5, 0.5), fontsize=10,
            ha="center", color="red", fontweight="bold")
ax.axis("off")

# ViT: global patch attention
ax = axes[1]
ax.set_xlim(0, 7); ax.set_ylim(0, 7); ax.set_aspect("equal")
colors_vit = plt.cm.Set3(np.linspace(0, 1, 49))
for idx, (i, j) in enumerate([(i, j) for i in range(7) for j in range(7)]):
    rect = plt.Rectangle((i, j), 1, 1, facecolor=colors_vit[idx],
                          edgecolor="gray", linewidth=0.5, alpha=0.7)
    ax.add_patch(rect)
center = (3.5, 3.5)
for i in range(7):
    for j in range(7):
        if (i, j) != (3, 3) and np.random.random() > 0.5:
            ax.annotate("", xy=(i + 0.5, j + 0.5), xytext=center,
                        arrowprops=dict(arrowstyle="->", color="red", lw=0.8, alpha=0.4))
ax.plot(*center, "r*", markersize=15, zorder=5)
ax.set_title("ViT: Global Patch Attention\n(every patch attends to every other)", fontsize=14, fontweight="bold")
ax.annotate("global from\nlayer 1", xy=(3.5, 0.5), fontsize=10,
            ha="center", color="red", fontweight="bold")
ax.axis("off")

plt.suptitle("How CNNs and ViTs Process Images", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout(); plt.show()
print("→ Teaching point: CNNs build up context slowly through stacked layers.")
print("  ViTs let every patch talk to every other patch from the very first layer.")

---

## Section 2: Data Tour — Radiology Images Meet Text

We use **ROCOv2** (Radiology Objects in COntext v2): ~80,000 radiology images from PubMed Central, each paired with a natural-language caption from the original medical paper. This dataset naturally combines **images** and **text**, making it ideal for both image classification and multimodal learning.

**Why ROCOv2?**
- Real clinical images (CT, MRI, X-ray, ultrasound, …)
- Natural medical captions (not just class labels)
- Supports classification *and* image-text alignment
- Published in *Scientific Data* (2024), freely available

In [ ]:
# ── Section 2: Load ROCOv2 Dataset ────────────────────────────────────────────
from datasets import load_dataset

print("Loading ROCOv2 from HuggingFace (this may take a minute on first run)...")
try:
    raw_ds = load_dataset("eltorio/ROCOv2-radiology", split="train")
except Exception:
    print("Primary source unavailable — trying fallback...")
    raw_ds = load_dataset("akahana/rocov2-full", split="train")

print(f"Loaded {len(raw_ds):,} examples")
print(f"Columns: {raw_ds.column_names}")
print(f"\nFirst example keys: {list(raw_ds[0].keys())}")

# ── Extract imaging modality from caption text ──
MODALITY_RULES = {
    "X-ray":       ["x-ray", "xray", "radiograph", "chest film", "plain film"],
    "CT":          ["ct ", "ct,", "ct.", "ct)", "computed tomography", "ct scan"],
    "MRI":         ["mri", "magnetic resonance", "mr image", "t1-weighted",
                    "t2-weighted", "flair", "t1w", "t2w", "dwi"],
    "Ultrasound":  ["ultrasound", "ultrasonograph", "sonograph", "echograph"],
    "Angiography": ["angiogra", "angiogram", "arteriogra"],
    "Mammography": ["mammogra"],
    "PET":         ["pet scan", "pet/ct", "pet-ct", "positron emission"],
    "Fluoroscopy": ["fluoroscop", "barium"],
    "Endoscopy":   ["endoscop", "colonoscop", "gastroscop"],
}

def extract_modality(text):
    text_lower = text.lower()
    for modality, keywords in MODALITY_RULES.items():
        if any(kw in text_lower for kw in keywords):
            return modality
    return "Other"

# Determine which column holds the caption
caption_col = "caption" if "caption" in raw_ds.column_names else raw_ds.column_names[1]
raw_ds = raw_ds.map(lambda x: {"modality": extract_modality(x[caption_col])})

modality_counts = Counter(raw_ds["modality"])
print("\nModality distribution (full dataset):")
for mod, cnt in modality_counts.most_common():
    print(f"  {mod:15s} {cnt:6,d}  ({100*cnt/len(raw_ds):.1f}%)")

In [ ]:
# ── Classroom subset: keep top modalities, sample 4000 train / 800 test ──
TOP_K_CLASSES = 5
top_modalities = [m for m, _ in modality_counts.most_common(TOP_K_CLASSES)]
print(f"Keeping top-{TOP_K_CLASSES} modalities: {top_modalities}")

filtered = raw_ds.filter(lambda x: x["modality"] in top_modalities)
label2id = {m: i for i, m in enumerate(sorted(top_modalities))}
id2label = {i: m for m, i in label2id.items()}
NUM_CLASSES = len(label2id)

filtered = filtered.map(lambda x: {"label": label2id[x["modality"]]})
filtered = filtered.shuffle(seed=SEED)

N_TRAIN, N_TEST = 4000, 800
train_ds = filtered.select(range(min(N_TRAIN, len(filtered))))
test_ds  = filtered.select(range(N_TRAIN, min(N_TRAIN + N_TEST, len(filtered))))

print(f"\nClassroom subset: {len(train_ds)} train, {len(test_ds)} test")
print(f"Classes ({NUM_CLASSES}): {label2id}")

# Quick peek at one example
ex = train_ds[0]
image_col = "image" if "image" in train_ds.column_names else train_ds.column_names[0]
print(f"\nExample caption: {ex[caption_col][:200]}...")
print(f"Modality: {ex['modality']}  |  Label: {ex['label']}")

In [ ]:
# Viz 2 — Image gallery: 4×4 grid with captions (diverse modalities)
sample_indices = []
for lbl in range(NUM_CLASSES):
    idxs = [i for i, x in enumerate(train_ds) if x["label"] == lbl]
    sample_indices.extend(idxs[:3])
sample_indices = sample_indices[:16]
np.random.shuffle(sample_indices)

gallery_imgs = []
gallery_titles = []
for i in sample_indices[:16]:
    ex = train_ds[i]
    img = ex[image_col]
    if not isinstance(img, Image.Image):
        img = Image.open(img).convert("RGB")
    elif img.mode != "RGB":
        img = img.convert("RGB")
    gallery_imgs.append(img)
    cap = ex[caption_col][:80] + "..." if len(ex[caption_col]) > 80 else ex[caption_col]
    gallery_titles.append(f"[{ex['modality']}] {cap}")

show_image_grid(gallery_imgs, gallery_titles, ncols=4, figsize=(18, 14))
print("→ Teaching point: Each image comes with a natural-language caption from its")
print("  medical paper. This pairing of images + text is what enables multimodal learning.")

In [ ]:
# Viz 3 — Imaging modality distribution
fig, ax = plt.subplots(figsize=(10, 5))
labels_list = [id2label[x["label"]] for x in train_ds]
counts = Counter(labels_list)
mods = sorted(counts.keys(), key=lambda m: counts[m], reverse=True)
vals = [counts[m] for m in mods]
colors = sns.color_palette("Set2", len(mods))
bars = ax.bar(mods, vals, color=colors, edgecolor="gray", linewidth=0.5)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            str(v), ha="center", fontsize=11, fontweight="bold")
ax.set_xlabel("Imaging Modality"); ax.set_ylabel("Count")
ax.set_title("Class Distribution in Training Set")
plt.tight_layout(); plt.show()

In [ ]:
# Viz 4 — Caption length distribution
cap_lengths = [len(x[caption_col].split()) for x in train_ds]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(cap_lengths, bins=40, color="steelblue", edgecolor="white", alpha=0.8)
ax.axvline(np.median(cap_lengths), color="coral", linestyle="--", linewidth=2,
           label=f"Median = {np.median(cap_lengths):.0f} words")
ax.set_xlabel("Caption Length (words)"); ax.set_ylabel("Count")
ax.set_title("Distribution of Caption Lengths")
ax.legend(fontsize=12)
plt.tight_layout(); plt.show()
print(f"Caption stats: min={min(cap_lengths)}, max={max(cap_lengths)}, "
      f"mean={np.mean(cap_lengths):.1f}, median={np.median(cap_lengths):.0f}")

In [ ]:
# Viz 5 — Top 25 medical terms in captions
STOP_WORDS = {"the", "a", "an", "of", "in", "and", "is", "to", "with", "was",
              "for", "on", "are", "that", "this", "from", "by", "at", "or", "as",
              "be", "it", "were", "been", "which", "has", "have", "had", "not",
              "but", "its", "can", "also", "into", "than", "no", "our", "their",
              "we", "after", "before", "between", "during", "each", "other", "both",
              "image", "figure", "show", "shows", "showing", "shown", "see", "case"}

all_words = []
for ex in train_ds:
    words = re.findall(r"[a-z]{3,}", ex[caption_col].lower())
    all_words.extend(w for w in words if w not in STOP_WORDS)

word_freq = Counter(all_words).most_common(25)
words_top, freqs_top = zip(*word_freq)

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(range(len(words_top)), freqs_top, color="steelblue", edgecolor="white")
ax.set_yticks(range(len(words_top)))
ax.set_yticklabels(words_top, fontsize=11)
ax.invert_yaxis()
ax.set_xlabel("Frequency"); ax.set_title("Top 25 Medical Terms in Captions")
plt.tight_layout(); plt.show()
print("→ Teaching point: The captions are rich in medical terminology — anatomy,")
print("  imaging findings, and diagnoses. This text is what we'll pair with images later.")

---

## Section 3: Image Tokenization — Seeing in Patches

This is the central insight of ViT. Just as NLP models split text into tokens, ViT splits an image into **patches** — fixed-size non-overlapping squares. Each patch becomes a "visual word" that the transformer processes.

For a 224×224 image with 16×16 patches:
- Number of patches = (224 / 16) × (224 / 16) = **14 × 14 = 196 patches**
- Each patch is 16 × 16 × 3 (RGB) = **768 values**
- A linear projection maps these 768 values to an embedding vector

```
224×224 image                     196 patch tokens
┌──┬──┬──┬──┬──┐                ┌───┐
│  │  │  │  │  │   flatten +    │ 1 │ → [768-dim vector]
├──┼──┼──┼──┼──┤   project      │ 2 │ → [768-dim vector]
│  │  │  │  │  │  ──────────→   │...│
├──┼──┼──┼──┼──┤                │196│ → [768-dim vector]
│  │  │  │  │  │                └───┘
└──┴──┴──┴──┴──┘
  14×14 grid                     + [CLS] token at position 0
```

The transformer then sees **197 tokens** (196 patches + 1 CLS token), exactly like processing a 197-word sentence.

In [ ]:
# Viz 6 — Patch grid overlay: see the 14×14 patch boundaries on a real image
demo_img = train_ds[0][image_col]
if not isinstance(demo_img, Image.Image):
    demo_img = Image.open(demo_img)
demo_img = demo_img.convert("RGB").resize((IMG_SIZE, IMG_SIZE))
demo_arr = np.array(demo_img)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Original image
axes[0].imshow(demo_arr); axes[0].set_title("Original Image (224×224)", fontsize=13)
axes[0].axis("off")

# Image with grid overlay
axes[1].imshow(demo_arr)
for i in range(0, IMG_SIZE + 1, PATCH_SIZE):
    axes[1].axhline(y=i, color="red", linewidth=0.8, alpha=0.7)
    axes[1].axvline(x=i, color="red", linewidth=0.8, alpha=0.7)
axes[1].set_title(f"Patch Grid ({IMG_SIZE//PATCH_SIZE}×{IMG_SIZE//PATCH_SIZE} = {NUM_PATCHES} patches)", fontsize=13)
axes[1].axis("off")

# Numbered patches (first 5×5 corner)
axes[2].imshow(demo_arr[:5*PATCH_SIZE, :5*PATCH_SIZE])
for pi in range(5):
    for pj in range(5):
        y, x = pi * PATCH_SIZE + PATCH_SIZE//2, pj * PATCH_SIZE + PATCH_SIZE//2
        idx = pi * (IMG_SIZE // PATCH_SIZE) + pj
        axes[2].text(x, y, str(idx), ha="center", va="center",
                     fontsize=9, color="yellow", fontweight="bold",
                     bbox=dict(boxstyle="round,pad=0.15", facecolor="black", alpha=0.6))
for i in range(0, 5*PATCH_SIZE + 1, PATCH_SIZE):
    axes[2].axhline(y=i, color="red", linewidth=1)
    axes[2].axvline(x=i, color="red", linewidth=1)
axes[2].set_title("Zoomed: Patch Indices (top-left corner)", fontsize=13)
axes[2].axis("off")

plt.suptitle("Step 1: Split Image into Patches", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout(); plt.show()
print(f"→ Each 16×16 patch is one 'token'. The transformer sees {NUM_PATCHES} tokens,")
print(f"  just like processing a {NUM_PATCHES}-word sentence.")

In [ ]:
# Viz 7 — Grid of all 196 individual patches (what each "token" sees)
patches = []
for i in range(0, IMG_SIZE, PATCH_SIZE):
    for j in range(0, IMG_SIZE, PATCH_SIZE):
        patch = demo_arr[i:i+PATCH_SIZE, j:j+PATCH_SIZE]
        patches.append(patch)

grid_h, grid_w = IMG_SIZE // PATCH_SIZE, IMG_SIZE // PATCH_SIZE
fig, axes = plt.subplots(grid_h, grid_w, figsize=(14, 14))
for idx, ax in enumerate(axes.flatten()):
    ax.imshow(patches[idx])
    ax.axis("off")

plt.suptitle(f"All {NUM_PATCHES} Patches — Each One Is a 'Visual Token'",
             fontsize=15, fontweight="bold", y=1.01)
plt.subplots_adjust(wspace=0.05, hspace=0.05)
plt.show()
print("→ Teaching point: Each tiny 16×16 square is all the information one token carries.")
print("  The transformer's job is to integrate information across ALL tokens via attention.")

In [ ]:
# Viz 8 — Patch embedding weights: what does the linear projection learn?
# Load a pretrained ViT and visualize its Conv2d projection filters
from transformers import ViTModel

pretrained_vit = ViTModel.from_pretrained("google/vit-base-patch16-224")
proj_weight = pretrained_vit.embeddings.patch_embeddings.projection.weight.data.cpu()
# proj_weight shape: (768, 3, 16, 16) — 768 filters, each 3×16×16

fig, axes = plt.subplots(4, 8, figsize=(16, 8))
for idx, ax in enumerate(axes.flatten()):
    if idx < proj_weight.shape[0]:
        filt = proj_weight[idx].permute(1, 2, 0).numpy()  # (16, 16, 3)
        filt = (filt - filt.min()) / (filt.max() - filt.min() + 1e-8)
        ax.imshow(filt)
    ax.axis("off")

plt.suptitle("Learned Patch Embedding Filters (first 32 of 768)\n"
             "Each filter extracts a different visual feature from a 16×16 patch",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout(); plt.show()
print("→ These learned filters look like edge detectors, color analyzers, and texture")
print("  extractors — similar to CNN first-layer filters but learned via the ViT objective.")

del pretrained_vit
torch.cuda.empty_cache() if torch.cuda.is_available() else None

### Position Embeddings — Teaching the Transformer About Space

Without position embeddings, the transformer has **no idea** where each patch is in the image. Patch 0 (top-left) and Patch 195 (bottom-right) would look identical structurally.

ViT adds a **learnable position embedding** to each patch token. After training, nearby patches learn similar position embeddings, encoding 2D spatial structure into 1D positional information.

In [ ]:
# Viz 9 — Position embedding similarity heatmap
from transformers import ViTModel

pretrained_vit = ViTModel.from_pretrained("google/vit-base-patch16-224")
pos_embed = pretrained_vit.embeddings.position_embeddings.data.cpu()  # (1, 197, 768)
pos_embed = pos_embed.squeeze(0)  # (197, 768) — includes CLS at position 0

# Cosine similarity between all patch position embeddings (skip CLS)
patch_pos = pos_embed[1:]  # (196, 768)
patch_pos_norm = F.normalize(patch_pos, dim=-1)
sim_matrix = (patch_pos_norm @ patch_pos_norm.T).numpy()  # (196, 196)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Full similarity matrix
im = axes[0].imshow(sim_matrix, cmap="RdBu_r", vmin=-0.3, vmax=1.0)
axes[0].set_title("Position Embedding Similarity\n(196 patches × 196 patches)", fontsize=13)
axes[0].set_xlabel("Patch Index"); axes[0].set_ylabel("Patch Index")
plt.colorbar(im, ax=axes[0], shrink=0.8)

# Similarity of one patch to all others, reshaped as 14×14 spatial map
reference_patches = [0, 7, 98, 105, 195]  # corners and center
grid_h = IMG_SIZE // PATCH_SIZE
axes[1].set_title("Similarity to Reference Patches\n(reshaped to 14×14 grid)", fontsize=13)
for k, ref_idx in enumerate([0, 98, 195]):
    sim_map = sim_matrix[ref_idx].reshape(grid_h, grid_h)
    ax_inset = axes[1].inset_axes([k * 0.35, 0.05, 0.3, 0.9])
    ax_inset.imshow(sim_map, cmap="RdBu_r", vmin=-0.3, vmax=1.0)
    row, col = ref_idx // grid_h, ref_idx % grid_h
    ax_inset.plot(col, row, "k*", markersize=12)
    pos_name = ["Top-left", "Center", "Bottom-right"][k]
    ax_inset.set_title(f"Ref: {pos_name}\n(patch {ref_idx})", fontsize=10)
    ax_inset.axis("off")
axes[1].axis("off")

plt.tight_layout(); plt.show()
print("→ Teaching point: Nearby patches learn similar position embeddings, creating a")
print("  smooth 2D spatial map. The model 'knows' where each patch is without being told.")

del pretrained_vit
torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [ ]:
# Viz 10 — Side-by-side: NLP tokenization vs ViT tokenization
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# NLP tokenization
ax = axes[0]
ax.set_xlim(0, 10); ax.set_ylim(0, 3); ax.axis("off")
ax.set_title("NLP: Text → Tokens", fontsize=14, fontweight="bold")
sentence = "The chest X-ray shows bilateral opacity"
words = sentence.split()
colors_nlp = plt.cm.Pastel1(np.linspace(0, 0.8, len(words)))
ax.text(5, 2.6, f'"{sentence}"', ha="center", fontsize=11, style="italic")
for i, (w, c) in enumerate(zip(words, colors_nlp)):
    x = 0.5 + i * (9.0 / len(words))
    rect = plt.Rectangle((x, 1.2), 9.0/len(words) - 0.15, 0.7,
                          facecolor=c, edgecolor="gray", linewidth=1)
    ax.add_patch(rect)
    ax.text(x + (9.0/len(words) - 0.15)/2, 1.55, w, ha="center", va="center", fontsize=9)
ax.annotate("", xy=(5, 1.15), xytext=(5, 2.4),
            arrowprops=dict(arrowstyle="->", color="steelblue", lw=2))
ax.text(5, 0.6, "→ Sequence of token embeddings", ha="center", fontsize=11, color="steelblue")

# ViT tokenization
ax = axes[1]
ax.set_xlim(0, 10); ax.set_ylim(0, 3); ax.axis("off")
ax.set_title("ViT: Image → Patch Tokens", fontsize=14, fontweight="bold")
n_show = 7
colors_vit = plt.cm.Set3(np.linspace(0, 0.8, n_show))
# Mini image grid
for i in range(4):
    for j in range(4):
        rect = plt.Rectangle((3.5 + j*0.5, 2.0 + (3-i)*0.25, ), 0.48, 0.23,
                              facecolor=plt.cm.Set3(((i*4+j)/16)),
                              edgecolor="gray", linewidth=0.5)
        ax.add_patch(rect)
ax.annotate("", xy=(5, 1.15), xytext=(5, 1.95),
            arrowprops=dict(arrowstyle="->", color="coral", lw=2))
for i in range(n_show):
    x = 0.8 + i * 1.25
    rect = plt.Rectangle((x, 0.2), 1.1, 0.7,
                          facecolor=colors_vit[i], edgecolor="gray", linewidth=1)
    ax.add_patch(rect)
    ax.text(x + 0.55, 0.55, f"P{i}", ha="center", va="center", fontsize=10)
ax.text(9.5, 0.55, "…", fontsize=14)
ax.text(5, 1.55, "flatten + project each patch", ha="center", fontsize=10, color="coral")

plt.suptitle("The Same Idea: Split Input → Embed → Feed to Transformer",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout(); plt.show()
print("→ The transformer doesn't know it's looking at an image. It just sees a sequence")
print("  of vectors, exactly like words. This is why the SAME architecture works for both.")

---

## Section 4: Building a Vision Transformer from Scratch

We now build a complete ViT, piece by piece. The architecture has four main components:

```
┌──────────────────────────────────────────────────────────────────┐
│ 1. PatchEmbedding     — Conv2d splits image into patch vectors  │
│ 2. MultiHeadAttention — Each patch attends to every other patch │
│ 3. TransformerBlock   — Attention + MLP + residual connections  │
│ 4. VisionTransformer  — Stack blocks, add [CLS], classify      │
└──────────────────────────────────────────────────────────────────┘
```

We use a **small model** (256-dim, 6 layers, 8 heads) that trains in minutes on a T4 GPU. The real `google/vit-base` uses 768-dim, 12 layers, 12 heads — same structure, just bigger.

In [ ]:
# ── Building Block 1: Patch Embedding ─────────────────────────────────────────

class PatchEmbedding(nn.Module):
    """Split an image into patches and project each to an embedding vector.

    Uses Conv2d with kernel_size=stride=patch_size — a clean trick that
    performs non-overlapping patch extraction and linear projection in one op.
    """
    def __init__(self, img_size=224, patch_size=16, in_channels=3, embed_dim=256):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim,
                              kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        # x: (B, 3, 224, 224) → (B, embed_dim, 14, 14)
        x = self.proj(x)
        # (B, embed_dim, 14, 14) → (B, 196, embed_dim)
        x = x.flatten(2).transpose(1, 2)
        return x


# Quick test
_pe = PatchEmbedding()
_dummy = torch.randn(2, 3, 224, 224)
print(f"Input:  {_dummy.shape}")
print(f"Output: {_pe(_dummy).shape}  (batch, num_patches={_pe.num_patches}, embed_dim=256)")

In [ ]:
# ── Building Block 2: Multi-Head Self-Attention ──────────────────────────────

class MultiHeadSelfAttention(nn.Module):
    """Standard multi-head self-attention with extractable attention weights.

    We store attention weights so we can visualize them later —
    which patches attend to which other patches?
    """
    def __init__(self, embed_dim=256, num_heads=8, dropout=0.0):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.attn_drop = nn.Dropout(dropout)
        self.attn_weights = None

    def forward(self, x):
        B, N, C = x.shape
        # Compute Q, K, V in one matrix multiply
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)  # (3, B, heads, N, head_dim)
        q, k, v = qkv.unbind(0)

        # Scaled dot-product attention
        attn = (q @ k.transpose(-2, -1)) * self.scale  # (B, heads, N, N)
        attn = attn.softmax(dim=-1)
        self.attn_weights = attn.detach()
        attn = self.attn_drop(attn)

        # Apply attention to values
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        return x


# Quick test
_mhsa = MultiHeadSelfAttention(embed_dim=256, num_heads=8)
_tokens = torch.randn(2, 197, 256)
print(f"Input:  {_tokens.shape}")
print(f"Output: {_mhsa(_tokens).shape}")
print(f"Attention weights: {_mhsa.attn_weights.shape}  (batch, heads, seq, seq)")